# Unit 6 Exercise: A Thread-Safe Bank Account

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 6.

The seminar asks for a simple multi-threaded application, such as a bank account with concurrent deposits and withdrawals, made thread safe with `threading` locks. This notebook first separates concurrency from parallelism with a small benchmark, then shows each problem the unit names, a race condition and a deadlock, together with its fix, and finally the two other synchronisation tools the seminar mentions, a monitor (condition variable) and a semaphore.

The code is in `bank_account.py` and the tests are in `test_bank_account.py`. Only the standard library is used.

In [1]:
import dis
import os
import platform
import sys
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from decimal import Decimal

import bank_account
from bank_account import (BankAccount, DeadlockDetected, InsufficientFunds, TellerDesk,
                          UnsafeAccount, count_primes, naive_transfer)

print("Python", platform.python_version(), "| CPU cores:", os.cpu_count())

Python 3.13.18 | CPU cores: 4


## 1. Concurrency is not parallelism

McDonald (2022, ch. 17) follows Rob Pike in defining concurrency as the composition of several tasks and parallelism as running several tasks at the same time. In CPython the difference is concrete: threads share one interpreter and its Global Interpreter Lock (GIL), so only one of them executes Python bytecode at a time, while each process has its own interpreter and its own GIL (Nguyen, 2022, ch. 8 and 15).

The benchmark runs the same CPU-bound job, counting primes by trial division, four times: one after another, in four threads and in four processes.

In [2]:
JOBS = [60_000] * 4

def timed(label, run):
    start = time.perf_counter()
    results = run()
    seconds = time.perf_counter() - start
    print(f"{label:<22} {seconds:5.2f} s   results {results}")
    return seconds

if __name__ == "__main__":
    sequential = timed("one after another", lambda: [count_primes(n) for n in JOBS])
    with ThreadPoolExecutor(max_workers=4) as pool:
        threaded = timed("four threads", lambda: list(pool.map(count_primes, JOBS)))
    with ProcessPoolExecutor(max_workers=4) as pool:
        parallel = timed("four processes", lambda: list(pool.map(count_primes, JOBS)))
    print(f"\nspeed-up with threads   x{sequential / threaded:.1f}")
    print(f"speed-up with processes x{sequential / parallel:.1f}")

one after another       0.28 s   results [6057, 6057, 6057, 6057]


four threads            0.30 s   results [6057, 6057, 6057, 6057]


four processes          0.09 s   results [6057, 6057, 6057, 6057]

speed-up with threads   x0.9
speed-up with processes x3.1


The threads give no speed-up on CPU-bound work, because they take turns with the GIL; the processes do, at the cost of starting interpreters and copying arguments and results between them. Threads still help when the work is waiting rather than computing, which is why Nguyen (2022, ch. 9) uses them for web requests. For a bank account the point is different: requests arrive concurrently whether or not they run in parallel, so the account has to be correct under any interleaving.

## 2. A race condition that the tests nearly missed

McDonald (2022, ch. 17) shows that `count += 1` is several bytecode instructions, a read, an add and a write, so a thread switch between them can lose an update.

In [3]:
def deposit_one(account):
    account.balance += 1

dis.dis(deposit_one)

  1           RESUME                   0

  2           LOAD_FAST                0 (account)
              COPY                     1
              LOAD_ATTR                0 (balance)
              LOAD_CONST               1 (1)
              BINARY_OP               13 (+=)
              SWAP                     2
              STORE_ATTR               0 (balance)
              RETURN_CONST             0 (None)


Yet on this interpreter the obvious stress test does not reproduce the loss. Eight threads each add 1 ten thousand times, even with the switch interval set as low as it will go:

In [4]:
class PlusEqualsAccount:
    def __init__(self):
        self.balance = 0

def hammer(account, action, threads=8, times=10_000):
    workers = [threading.Thread(target=lambda: [action(account) for _ in range(times)])
               for _ in range(threads)]
    for w in workers: w.start()
    for w in workers: w.join()
    return account.balance

previous = sys.getswitchinterval()
sys.setswitchinterval(1e-6)
results = [hammer(PlusEqualsAccount(), deposit_one) for _ in range(5)]
sys.setswitchinterval(previous)
print("expected 80000, got", results)

expected 80000, got [80000, 80000, 80000, 80000, 80000]


CPython only switches threads at certain points in the bytecode, and on this version the read and write of `+=` happen not to be split. That is a property of one interpreter release, not a guarantee of the language, and the free-threaded build introduced in Python 3.13 removes the GIL altogether (Gross, 2023). A passing test here proves nothing. `UnsafeAccount` makes the gap visible by doing what real code does between the read and the write: a little work, represented by `time.sleep(0)`.

In [5]:
unsafe = UnsafeAccount()
print("UnsafeAccount, 8 threads x 1000 deposits:",
      hammer(unsafe, lambda a: a.deposit(1), times=1000), "expected 8000")

safe = BankAccount("Alice")
print("BankAccount,   8 threads x 1000 deposits:",
      hammer(safe, lambda a: a.deposit(1), times=1000), "expected 8000")

UnsafeAccount, 8 threads x 1000 deposits: 1013.00 expected 8000
BankAccount,   8 threads x 1000 deposits: 8000.00 expected 8000


### Time of check to time of use

The same gap is a security bug in a withdrawal. Nguyen (2022, ch. 14) describes the time-of-check to time-of-use (TOCTTOU) race in authentication; in a bank it is a double spend. A barrier makes the interleaving certain: both threads check the balance, then both debit it.

In [6]:
def run_pair(first, second):
    errors = []
    def guard(fn):
        try:
            fn()
        except Exception as e:
            errors.append(type(e).__name__)
    threads = [threading.Thread(target=guard, args=(f,)) for f in (first, second)]
    for t in threads: t.start()
    for t in threads: t.join(5)
    return errors

barrier = threading.Barrier(2, timeout=2)
unsafe = UnsafeAccount(100, pause=barrier.wait)
errors = run_pair(lambda: unsafe.withdraw(100), lambda: unsafe.withdraw(100))
print(f"UnsafeAccount: balance {unsafe.balance}, refused {errors}")

safe = BankAccount("Bob", 100)
errors = run_pair(lambda: safe.withdraw(100), lambda: safe.withdraw(100))
print(f"BankAccount:   balance {safe.balance}, refused {errors}")

UnsafeAccount: balance -100.00, refused []
BankAccount:   balance 0.00, refused ['InsufficientFunds']


The fix is mutual exclusion: `BankAccount.withdraw` checks and debits while holding the account's lock, so the two steps become one. The lock and the balance are private to the same object and every public method takes the lock, which is the object-oriented answer to Nguyen's warning that 'locks do not lock anything' (2022, ch. 14): callers never see the lock, so they cannot forget it.

In [7]:
import inspect
print(inspect.getsource(BankAccount.withdraw))

    def withdraw(self, amount: AmountLike) -> Decimal:
        """Remove ``amount``, checking and debiting while holding the lock."""
        value = to_amount(amount)
        with self._lock:
            self._require(value)
            self._debit(value, "withdrawal")
            return self._balance



## 3. A deadlock and its fix

A transfer needs both accounts' locks. `naive_transfer` takes the source lock and then the target lock, so a transfer from A to B and one from B to A can each hold one lock and wait for the other. All four Coffman conditions hold: the locks are exclusive, each thread holds one while waiting for another, neither can be taken away, and the waits form a cycle (Coffman, Elphick and Shoshani, 1971). A barrier after the first lock forces this; the timeout is there only so that the notebook finishes.

In [8]:
a, b = BankAccount("Carol", 500), BankAccount("Dan", 500)
barrier = threading.Barrier(2, timeout=2)
start = time.perf_counter()
errors = run_pair(
    lambda: naive_transfer(a, b, 10, after_first_lock=barrier.wait, timeout=0.5),
    lambda: naive_transfer(b, a, 10, after_first_lock=barrier.wait, timeout=0.5))
print(f"naive_transfer: {errors} after {time.perf_counter() - start:.2f} s")
print(f"balances afterwards: {a.balance}, {b.balance}")

naive_transfer: ['DeadlockDetected'] after 0.50 s
balances afterwards: 490.00, 510.00


Both threads took their first lock and waited for the second. The first to time out gave up and released its lock, which let the other thread finish its transfer, so one transfer of 10 went through and one was abandoned. A timeout is McDonald's (2022, ch. 17) first remedy and it does break the hold-and-wait condition, but it turns a hang into a failed transaction that the caller has to retry, and under steady load two threads that keep timing out and retrying can livelock.

`BankAccount.transfer_to` removes the circular wait by ranking the resources, as Nguyen (2022, ch. 12) does with the dining philosophers: whichever way the money moves, the lower-numbered account is locked first. Nguyen sorts locks by `id()`; account numbers do the same job and also make the order readable in logs.

In [9]:
print(inspect.getsource(BankAccount.transfer_to))

    def transfer_to(self, other: BankAccount, amount: AmountLike) -> None:
        """Move ``amount`` from this account to ``other`` atomically.

        Both locks are needed. Taking them in account-number order, whichever
        direction the money moves, means no two transfers can each hold the
        lock the other wants, which removes the circular wait that a deadlock
        needs.
        """
        value = to_amount(amount)
        if other is self:
            # Lock is not reentrant: locking the same account twice would
            # leave this thread waiting for itself.
            raise ValueError("cannot transfer an account to itself")
        first, second = sorted((self, other), key=lambda account: account.number)
        with first._lock, second._lock:
            self._require(value)
            self._debit(value, f"transfer to {other.number}")
            other._credit(value, f"transfer from {self.number}")
            other._funds_added.notify_all()



In [10]:
def repeat(times, action):
    def run():
        for _ in range(times):
            action()
    return run

a, b = BankAccount("Carol", 5000), BankAccount("Dan", 5000)
start = time.perf_counter()
print("errors:", run_pair(repeat(5000, lambda: a.transfer_to(b, 1)),
                          repeat(5000, lambda: b.transfer_to(a, 1))))
print(f"2 x 5000 opposite transfers in {time.perf_counter() - start:.2f} s; "
      f"total {a.balance + b.balance} (expected 10000.00)")

ring = [BankAccount(f"P{i}", 1000) for i in range(5)]
workers = [threading.Thread(target=repeat(1000, lambda i=i: ring[i].transfer_to(ring[(i + 1) % 5], 1)))
           for i in range(5)]
for w in workers: w.start()
for w in workers: w.join(10)
print("dining-philosophers ring of five accounts finished:",
      not any(w.is_alive() for w in workers), "| total", sum(r.balance for r in ring))

errors: []
2 x 5000 opposite transfers in 0.04 s; total 10000.00 (expected 10000.00)
dining-philosophers ring of five accounts finished: True | total 5000.00


One deadlock needs no second thread. A `threading.Lock` is not reentrant, so a thread that already holds it and asks again waits for itself (McDonald, 2022, ch. 17). That is why `transfer_to` refuses a transfer to the same account, and why code inside the lock calls private helpers rather than the public methods.

In [11]:
lock = threading.Lock()
lock.acquire()
print("Lock acquired twice by the same thread:", lock.acquire(timeout=0.2))
lock.release()

rlock = threading.RLock()
rlock.acquire()
print("RLock acquired twice by the same thread:", rlock.acquire(timeout=0.2))
rlock.release(); rlock.release()

try:
    acct = BankAccount("Self", 10)
    acct.transfer_to(acct, 1)
except ValueError as error:
    print("transfer to self:", error)

Lock acquired twice by the same thread: False
RLock acquired twice by the same thread: True
transfer to self: cannot transfer an account to itself


## 4. Locks, semaphores and monitors

A lock admits one thread. A semaphore admits up to *n*: `TellerDesk` serves at most three customers at once with a `BoundedSemaphore`. The semaphore limits how many threads are inside, but it does not protect the data they share, so the busy count still needs its own lock.

In [12]:
desk = TellerDesk(tellers=3)
customers = [threading.Thread(target=lambda: desk.serve(lambda: time.sleep(0.05))) for _ in range(12)]
start = time.perf_counter()
for c in customers: c.start()
for c in customers: c.join()
print(f"12 customers, 3 tellers: most served at once {desk.peak}, "
      f"took {time.perf_counter() - start:.2f} s (12 x 0.05 s / 3 = 0.20 s)")

12 customers, 3 tellers: most served at once 3, took 0.20 s (12 x 0.05 s / 3 = 0.20 s)


A monitor combines the data, the lock and the conditions to wait on in one object, the structure Hoare (1974) proposed. `BankAccount` is one: `withdraw_when_funded` waits on a condition variable built on the account's own lock. While it waits it releases the lock, so a deposit can get in, and it re-checks the balance each time it is woken.

In [13]:
account = BankAccount("Erin")
results = []
waiter = threading.Thread(target=lambda: results.append(account.withdraw_when_funded(30, timeout=5)))
waiter.start()
time.sleep(0.1)
print("waiting, balance", account.balance, "| result so far", results)
account.deposit(50)
waiter.join()
print("after a deposit of 50:", results)

try:
    BankAccount("Finn", 10).withdraw_when_funded(20, timeout=0.2)
except InsufficientFunds as error:
    print("no deposit arrives:", error)

waiting, balance 0.00 | result so far []
after a deposit of 50: [Decimal('20.00')]


no deposit arrives: account 14 did not reach 20.00 within 0.2 s


## 5. What the lock costs

McDonald (2022, ch. 20) warns that 'all claims of performance are mythical until proven otherwise'. `timeit` measures the price of the lock on an uncontended deposit, compared with the unsafe account with its artificial pause removed.

In [14]:
from timeit import timeit

fast_unsafe = UnsafeAccount(pause=lambda: None)
safe = BankAccount("Gail")
n = 200_000
unsafe_us = timeit(lambda: fast_unsafe.deposit(1), number=n) / n * 1e6
safe_us = timeit(lambda: safe.deposit(1), number=n) / n * 1e6
print(f"UnsafeAccount.deposit  {unsafe_us:.2f} microseconds")
print(f"BankAccount.deposit    {safe_us:.2f} microseconds (lock, history entry and notify)")

UnsafeAccount.deposit  0.74 microseconds
BankAccount.deposit    1.93 microseconds (lock, history entry and notify)


The lock is cheap compared with any real work a bank does. The larger cost is the one Nguyen (2022, ch. 12 and 14) points out: code inside a lock runs one thread at a time, so the critical sections should be as short as possible.

## 6. The tests

The race and deadlock tests use barriers so that each failure happens every time rather than by chance, the stress tests use the smallest switch interval, and every thread is joined with a timeout, so a deadlock fails a test instead of hanging the suite.

In [15]:
import io
import unittest
import test_bank_account

stream = io.StringIO()
suite = unittest.defaultTestLoader.loadTestsFromModule(test_bank_account)
result = unittest.TextTestRunner(stream=stream, verbosity=1).run(suite)
print(f"Ran {result.testsRun} tests: {len(result.failures)} failures, {len(result.errors)} errors")

Ran 32 tests: 0 failures, 0 errors
